Analysing objects in 3D
---

*Introduction to Image Analysis Workshop*

*Stefania Marcotti (stefania.marcotti@crick.ac.uk)*

*Analysing objects in 3 dimensions*

*CC-BY-SA-4.0 license: creativecommons.org/licenses/by-sa/4.0/*

*Adapted from Tom Slater (slatert2@cardiff.ac.uk)*

*[Intro to building image analysis pipelines for 3D data with Python](https://github.com/RMS-DAIM/introduction-to-image-analysis/blob/main/Scripts/Jupyter/3d_analysis.ipynb)*

---

## Import libraries

In [1]:
import numpy as np
import matplotlib.pyplot as plt

from scipy import ndimage as ndi

import skimage

import pandas as pd

import napari

## Open image

We are going to use [a demo image from `skimage`](https://scikit-image.org/docs/stable/api/skimage.data.html#skimage.data.cells3d), which contains 3 dimensions and 2 channels (membrane, ch0, and nuclei, ch1). For this exercise, we are going to subset the data to only keep the nuclear channel.

In [3]:
import tqdm as notebook_tqdm

In [4]:
im_all = skimage.data.cells3d()

We can use the `shape` function to check image dimensionality

In [5]:
im_all.shape

(60, 2, 256, 256)

Remember that the convention is (C,Z,Y,X)! We need to transpose the first two dimensions

In [6]:
im_transpose = im_all.transpose(1,0,2,3)
im_transpose.shape

(2, 60, 256, 256)

We can visualise the dataset using `napari`. If we load the two channels separately, we can assign them different colormaps and names.

In [7]:
viewer = napari.Viewer()
viewer.add_image(im_transpose[0,], name='membranes', colormap='magenta', blending='additive')
viewer.add_image(im_transpose[1,], name='nuclei', colormap='green', blending='additive')

<Image layer 'nuclei' at 0x1a5b4e18c10>

We want to work with the nuclear channel for this exercise, therefore we'll need to subset the data accordingly. Note that the nuclear channel is the second one

In [8]:
im = im_transpose[1,]

In [9]:
im.shape

(60, 256, 256)

In [ ]:
viewer = napari.Viewer()
viewer.add_image(im, name='nuclei', colormap='gray', blending='additive')

## Use filters to suppress noise
We can apply a Gaussian blur to reduce noise and to help with segmentation quality

In [16]:
from skimage import measure
from skimage import filters
from skimage import segmentation
from skimage import io
from skimage import feature

In [12]:
im_gauss = filters.gaussian(im, sigma=2)

In [ ]:
viewer.add_image(im_gauss, name='nuclei_gauss', colormap='gray', blending='additive')

## Segment volumes
We can segment the filtered nuclei using one of the algorithms available in `scikit-image`

In [13]:
thresh = filters.threshold_otsu(im_gauss)

im_thresh = im_gauss > thresh

In [ ]:
viewer.add_labels(im_thresh, name='nuclei_binary', blending='additive')

## Count objects
We use <code>skimage.measure.label</code> to create a label image from the binary mask

In [14]:
# label objects and visualise the result
labels = measure.label(im_thresh)

# count the objects - find the maximum integer assigned to a label!
print('There are', labels.max(), 'objects in the image')

There are 23 objects in the image


In [ ]:
viewer.add_labels(labels, name='nuclei_labels', blending='additive')

We might be able to improve the quality of the segmentation with seeded watershedding. To calculate the seeds, we need:
* the distance transform of the image
* the local maxima of the distance transform
* an image with the labelled seeded markers

In [15]:
# calculate distance transform
distance = ndi.distance_transform_edt(im_thresh)

# find local maxima
coords = feature.peak_local_max(distance, min_distance=20, labels=labels)

# create an empty image with the same shape as the input image
mask = np.zeros(im.shape, dtype=bool)

# add the markers as True pixels in the coordinates stored in coords (this is a binary mask!)
mask[tuple(coords.T)] = True

# label each True pixel (this is a label mask!)
markers = measure.label(mask)

# run seeded watershed
labels_watershed = segmentation.watershed(-distance, markers, mask=im_thresh)

# count the objects - find the maximum integer assigned to a label!
print('There are', labels_watershed.max(), 'objects in the image')

NameError: name 'feature' is not defined

In [ ]:
viewer.add_labels(labels_watershed, name='nuclei_labels_watershed', blending='additive')

## Perform morphological quantification
We can use `regionprops` to look at the nuclear volume and centroids

In [ ]:
# measure properties
props = measure.regionprops_table(labels_watershed, properties=['label', 'area', 'centroid'])
props_df = pd.DataFrame(props)

props_df.head()

In [ ]:
# Plot a histogram of volumes
regions = measure.regionprops(labels_watershed)
volumes = [region.area for region in regions]

plt.hist(volumes, bins=40)
plt.xlabel('Volume (voxels)')
plt.ylabel('Count')
plt.title('Distribution of object volumes')

plt.tight_layout()